# 02 -- Paper Model Reconstruction

Instantiates the architecture scaffold, validates tensor/label/normalization mechanics, and demonstrates the annual training scheduler shape. No hyperparameter search happens in this notebook -- widths, dropout, loss, and all training hyperparameters are decision-gated per `docs/paper_model_architecture.md` and `docs/paper_loss_reconstruction_audit.md`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import PAPER_UNIVERSE
from src.tensors import build_tensor, build_paper_tensor
from src.labels import label_monday_open_to_friday_close, label_friday_close_to_friday_close, build_paper_labels
from src.normalization import fit_normalization, apply_normalization, assert_no_lookahead, fit_paper_normalization
from src.model import build_paper_model, N_OUTPUTS, N_HIDDEN_LAYERS
from src.training_schedule import AnnualScheduler
from src.decisions import PaperDecisionRequiredError

## Tensor shape validation

`X_t` is generic in `N` (the lookback decision) -- shown here with a placeholder N purely to validate mechanics (chronological ordering, no-lookahead), not as a resolution of `DECISION_REQUIRED_LOOKBACK_N`.

In [2]:
rng = np.random.default_rng(0)
toy_panel = pd.DataFrame(rng.normal(size=(30, 11)), columns=list(PAPER_UNIVERSE))
x_t = build_tensor(toy_panel, n=8, t_index=20, price_field='close', include_volume=False)
print('X_t shape (N=8 placeholder, l=11):', x_t.shape)
assert x_t.shape == (8, 11)

X_t shape (N=8 placeholder, l=11): (8, 11)


In [3]:
try:
    build_paper_tensor(toy_panel, t_index=20)
except PaperDecisionRequiredError as exc:
    print('Paper-faithful tensor construction blocked as expected:')
    print(exc)

Paper-faithful tensor construction blocked as expected:
[DECISION_REQUIRED_LOOKBACK_N] (Level 1) 'Input tensor history depth N' is not resolved in decisions/paper_decision_register.json. Required before: constructing the paper-faithful input tensor X_t. See docs/paper_source_audit.md and the decision register for source evidence and candidate options.


## Label mechanics

Both candidate target-interval definitions are implemented; the paper-facing entry point refuses to choose between them (`DECISION_REQUIRED_TARGET_RETURN_INTERVAL`).

In [4]:
print('Mon-open->Fri-close, +1.2% move:', label_monday_open_to_friday_close(100.0, 101.2))
print('Fri-close->Fri-close, +0.5% move:', label_friday_close_to_friday_close(100.0, 100.5))
try:
    build_paper_labels(price_panel=toy_panel)
except PaperDecisionRequiredError as exc:
    print('\nPaper-faithful label construction blocked as expected:')
    print(exc)

Mon-open->Fri-close, +1.2% move: 1
Fri-close->Fri-close, +0.5% move: 0

Paper-faithful label construction blocked as expected:
[DECISION_REQUIRED_TARGET_RETURN_INTERVAL] (Level 1) 'Exact date range the +1% label return is measured over' is not resolved in decisions/paper_decision_register.json. Required before: constructing paper-faithful labels y_{t+1} for training. See docs/paper_source_audit.md and the decision register for source evidence and candidate options.


## Normalization (lookahead-safety demonstration)

In [5]:
toy_series = toy_panel[['XLK']].to_numpy()
stats = fit_normalization(toy_series, fit_on_end_index=15)
assert_no_lookahead(stats, predict_row_index=16)  # OK
try:
    assert_no_lookahead(stats, predict_row_index=10)  # violation
except AssertionError as exc:
    print('Lookahead guard triggered as expected:', exc)

try:
    fit_paper_normalization(toy_series, current_week_index=15)
except PaperDecisionRequiredError as exc:
    print('\nPaper-faithful normalization scope blocked as expected:')
    print(exc)

Lookahead guard triggered as expected: Lookahead violation: normalization stats fit through row 15 but being applied to predict row 10 (must be fit strictly before the predicted row).

Paper-faithful normalization scope blocked as expected:
[DECISION_REQUIRED_NORMALIZATION_SCOPE] (Level 1) 'Scope over which zero-mean/unit-variance normalization stats are computed' is not resolved in decisions/paper_decision_register.json. Required before: fitting paper-faithful normalization statistics. See docs/paper_source_audit.md and the decision register for source evidence and candidate options.


## Architecture and training scheduler

`N_OUTPUTS` and `N_HIDDEN_LAYERS` are EXPLICIT paper facts (11 outputs, 4 hidden layers). Actually building/training the model is blocked pending the Level-1 decisions (hidden widths, dropout rate, custom financial loss, and all training hyperparameters).

In [6]:
print(f'N_OUTPUTS={N_OUTPUTS}, N_HIDDEN_LAYERS={N_HIDDEN_LAYERS}')

sched = AnnualScheduler()
print('Trading years:', sched.trading_years())
print('2012 training window:', sched.training_window_for_year(2012))

try:
    build_paper_model(n_history=8, n_features=11)
except PaperDecisionRequiredError as exc:
    print('\nPaper-faithful model construction blocked as expected:')
    print(exc)

N_OUTPUTS=11, N_HIDDEN_LAYERS=4
Trading years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
2012 training window: (2010, 2012)

Paper-faithful model construction blocked as expected:
[DECISION_REQUIRED_HIDDEN_WIDTHS] (Level 1) 'Units per hidden Dense layer (4 layers)' is not resolved in decisions/paper_decision_register.json. Required before: building the paper-faithful model. See docs/paper_source_audit.md and the decision register for source evidence and candidate options.


## Summary

Tensor/label/normalization mechanics and the annual-scheduler shape are fully implemented and tested. Actual model instantiation and training remain blocked on the Level-1 decisions enumerated in `decisions/paper_decision_register.json` -- this notebook demonstrates every blocking point explicitly rather than guessing past it.